# ISIC 2019 — EfficientNet-B0 Architecture

[PyTorch Data Tutorial](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html) 
https://isic-archive.s3.amazonaws.com/challenges/2019


ISIC 2019 data is downloaded and placed under `DATA_ROOT` below.

In [79]:
!pip install timm -q

In [80]:
import os, json, time, warnings
import wandb
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn

import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms

import timm
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore")
print("Imports OK")

Imports OK


In [81]:
import subprocess, zipfile

# PATHS 
DATA_ROOT = "/scratch/umw7eg/isic2019"
CKPT_DIR  = "/scratch/umw7eg/isic2019/checkpoints"
os.makedirs(DATA_ROOT, exist_ok=True)

BASE_URL = "https://isic-archive.s3.amazonaws.com/challenges/2019"

In [3]:
# Download CSVs & zips 
DOWNLOADS = [
    (f"{BASE_URL}/ISIC_2019_Training_GroundTruth.csv", f"{DATA_ROOT}/train_gt.csv"),
    (f"{BASE_URL}/ISIC_2019_Training_Metadata.csv",    f"{DATA_ROOT}/train_meta.csv"),
    (f"{BASE_URL}/ISIC_2019_Test_GroundTruth.csv",     f"{DATA_ROOT}/test_gt.csv"),
    
    (f"{BASE_URL}/ISIC_2019_Training_Input.zip",       f"{DATA_ROOT}/train_imgs.zip"),
    (f"{BASE_URL}/ISIC_2019_Test_Input.zip",           f"{DATA_ROOT}/test_imgs.zip"),
]

for url, dest in DOWNLOADS:
    print(f"Downloading {os.path.basename(dest)} ...")
    subprocess.run(
        ["curl", "--location", "--progress-bar",
         "--retry", "3", "--retry-delay", "5",
         "--output", dest, url],
        check=True,
    )

######################################################################## 100.0%
######################################################################## 100.0%
######################################################################## 100.0%


######################################################################## 100.0%


######################################################################## 100.0%


In [ ]:
# Unzip
for zip_path, extract_dir in [
    (f"{DATA_ROOT}/train_imgs.zip", DATA_ROOT),
    (f"{DATA_ROOT}/test_imgs.zip",  DATA_ROOT),
]:
    print(f"Extracting {os.path.basename(zip_path)} ...")
    
    # Sanity check
    size_mb = os.path.getsize(zip_path) / (1024**2)
    print(f"  File size: {size_mb:.1f} MB")
    if size_mb < 1:
        raise RuntimeError(f"File too small — download likely failed: {zip_path}")
    
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)
    
    os.remove(zip_path)
    print(f"  Done & removed {os.path.basename(zip_path)}")

In [82]:
# Path 
TRAIN_DIR  = os.path.join(DATA_ROOT, "ISIC_2019_Training_Input")
TEST_DIR   = os.path.join(DATA_ROOT, "ISIC_2019_Test_Input")
TRAIN_CSV  = os.path.join(DATA_ROOT, "train_gt.csv")
TEST_CSV   = os.path.join(DATA_ROOT, "test_gt.csv")
TRAIN_META = os.path.join(DATA_ROOT, "train_meta.csv")

print("Done! Data ready.")

Done! Data ready.


## Config for ablation studies

In [98]:
#os.makedirs(CKPT_DIR, exist_ok=True)

Config = {
    # model
    "architecture": "efficientnet_b0",  # "resnet50" | "mobilenetv3_small"
    "pretrained":   True,              # False = train from scratch ablation
    "freeze_bb":    False,             # True = head-only training ablation
    "loss_fn":      "weighted_ce",      # "ce" | "focal"
    "augmentation": "strong",           # "none" | "geometric" | "color" | "strong"
    "img_size":     224,
    "batch_size":   32,
    "epochs":       30,
    "lr":           1e-4,
    "val_split":    0.20,
    "patience":     10,                #stop epoch if no improvement for 10
    "seed":         42,                #set_seed for reproducibility 
    "classes":      ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"],
}
Config["num_classes"] = len(Config["classes"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Config: {Config}")

Device: cuda
Config: {'architecture': 'efficientnet_b0', 'pretrained': True, 'freeze_bb': False, 'loss_fn': 'weighted_ce', 'augmentation': 'strong', 'img_size': 224, 'batch_size': 32, 'epochs': 30, 'lr': 0.0001, 'val_split': 0.2, 'patience': 10, 'seed': 42, 'classes': ['MEL', 'NV', 'BCC', 'AK', 'BKL', 'DF', 'VASC', 'SCC'], 'num_classes': 8}


In [84]:
import random
def set_seed(seed=Config["seed"]):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
print(f"Seed set to {Config['seed']}")

Seed set to 42


In [85]:
#wandb login
#!echo $PATH
#os.environ["PATH"] = f"/home/umw7eg/.local/bin:{os.environ['PATH']}"

In [129]:
#Dataset 

class ISICDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df        = df.reset_index(drop=True)
        self.img_dir   = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(f"{self.img_dir}/{row['image']}.jpg").convert("RGB")
        
        if self.transform:
            img = self.transform(img)
        label = int(row["label"])
        return img, label

train_tfms = transforms.Compose([
    transforms.Resize((Config["img_size"], Config["img_size"])),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    #transforms.Normalize([0.485, 0.456, 0.406],
    #                     [0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((Config["img_size"], Config["img_size"])),
    transforms.ToTensor(),
    #transforms.Normalize([0.485, 0.456, 0.406],
    #                     [0.229, 0.224, 0.225]),
])

In [130]:
from torchvision import models

def build_model(num_classes=Config["num_classes"]):
    weights = models.EfficientNet_B0_Weights.DEFAULT if Config["pretrained"] else None
    model = models.efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

model = build_model(num_classes=Config["num_classes"]).to(device)
print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=8, bias=True)
)


In [131]:
import torch.nn.functional as F
from sklearn.metrics import (balanced_accuracy_score, confusion_matrix)

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [132]:
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)                        
            probs   = F.softmax(outputs, dim=1)
            preds   = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())       

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs  = np.array(all_probs)

    acc  = (all_preds == all_labels).mean()
    bacc = balanced_accuracy_score(all_labels, all_preds)

    # sensitivity & specificity per class
    cm = confusion_matrix(all_labels, all_preds, labels=list(range(Config["num_classes"])))
    sensitivity, specificity = [], []
    for i in range(Config["num_classes"]):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        sensitivity.append(tp / (tp + fn) if (tp + fn) > 0 else 0.0)
        specificity.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)

    # AUC per class
    auc_per_class = {}
    for i, cls in enumerate(Config["classes"]):
        try:
            auc_per_class[cls] = roc_auc_score((all_labels == i).astype(int), all_probs[:, i])
        except ValueError:
            auc_per_class[cls] = float("nan")

    return acc, bacc, np.mean(sensitivity), np.mean(specificity), auc_per_class

In [133]:
#training loop 

def train_model(train_df, val_df, num_classes):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_ds = ISICDataset(train_df, CONFIG["data_root"], train_tfms)
    val_ds = ISICDataset(val_df, CONFIG["data_root"], val_tfms)

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True, num_workers=4, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"], shuffle=False, num_workers=4, pin_memory=True)

    model = build_model(num_classes).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])
    criterion = nn.CrossEntropyLoss()

    best_acc = 0
    patience = 10
    no_improve = 0

    for epoch in range(CONFIG["epochs"]):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_acc = evaluate(model, val_loader, device)

        print(f"Epoch {epoch+1}: Loss={train_loss:.4f}, Val Acc={val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            no_improve = 0
            torch.save(model.state_dict(), "best_model.pth")
        else:
            no_improve += 1

        if no_improve >= patience:
            print("Early stopping triggered")
            break

    return model

In [145]:
train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

# Convert label column
train_df["label"] = train_df[Config["classes"]].values.argmax(axis=1)
test_df["label"]  = test_df[Config["classes"]].values.argmax(axis=1)

# Train/val split stratified by label...to be improved later by adding patient-level split with lesion_id(GroupShuffleSplit)
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(
    train_df,
    test_size=Config["val_split"],
    stratify=train_df["label"],
    random_state=Config["seed"],
)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 20264 | Val: 5067 | Test: 8238


In [157]:
counts        = train_df["label"].value_counts().sort_index()
CLASS_FREQ    = (counts / counts.sum()).to_dict()

print("Class frequencies:")

for name, df_ in [("TRAIN", train_df), ("VAL", val_df)]:
    labels = df_[Config["classes"]].values.argmax(axis=1)
    cnts   = np.bincount(labels, minlength=len(Config["classes"]))

    print(f"\n{name} ({len(df_)} images):")
    for i, c in enumerate(Config["classes"]):
        print(f"  {c:6s}: {cnts[i]:5d}  ({100*cnts[i]/len(df_):4.1f}%)")

Class frequencies:

TRAIN (20264 images):
  MEL   :  3618  (17.9%)
  NV    : 10300  (50.8%)
  BCC   :  2658  (13.1%)
  AK    :   694  ( 3.4%)
  BKL   :  2099  (10.4%)
  DF    :   191  ( 0.9%)
  VASC  :   202  ( 1.0%)
  SCC   :   502  ( 2.5%)

VAL (5067 images):
  MEL   :   904  (17.8%)
  NV    :  2575  (50.8%)
  BCC   :   665  (13.1%)
  AK    :   173  ( 3.4%)
  BKL   :   525  (10.4%)
  DF    :    48  ( 0.9%)
  VASC  :    51  ( 1.0%)
  SCC   :   126  ( 2.5%)


In [158]:
# 10% subset TO REMOVE LATER
#train_df = train_df.sample(frac=0.1, random_state=Config["seed"]).reset_index(drop=True)
#val_df   = val_df.sample(frac=0.1,   random_state=Config["seed"]).reset_index(drop=True)
#test_df  = test_df.sample(frac=0.1,  random_state=Config["seed"]).reset_index(drop=True)

print(f"Subset — train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")

# Datasets
train_ds = ISICDataset(train_df, TRAIN_DIR, train_tfms)
val_ds   = ISICDataset(val_df,   TRAIN_DIR, val_tfms)
test_ds  = ISICDataset(test_df,  TEST_DIR,  val_tfms)  # reuse val_transforms

# Dataloaders
train_loader = DataLoader(
    train_ds,
    batch_size=Config["batch_size"],
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=Config["batch_size"],
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=Config["batch_size"],
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

Subset — train: 20264, val: 5067, test: 8238


In [ ]:
#full training loop 

model     = build_model().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=Config["lr"])
loss_fn   = nn.CrossEntropyLoss()
scheduler = CosineAnnealingLR(optimizer, T_max=Config["epochs"], eta_min=1e-6)

best_bacc, best_epoch, no_improve = -1, 0, 0
history   = []
ckpt_path = os.path.join(CKPT_DIR, "efficientnet_b0_best.pt")

print(f"Training | pretrained={Config['pretrained']} | loss={Config['loss_fn']}\n")
t0 = time.time()

for epoch in range(1, Config["epochs"] + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, loss_fn, device)
    acc, bacc, sensitivity, specificity, auc_per_class = evaluate(model, val_loader, device)

    history.append({
        "epoch":      epoch,
        "train_loss": train_loss,
        "val_acc":    acc,
        "val_bacc":   bacc,
        "val_sens":   np.mean(sensitivity),
        "val_spec":   np.mean(specificity),
        "val_auc":    np.mean(list(auc_per_class.values())),
    })

    print(f"Epoch {epoch:3d}/{Config['epochs']} | loss={train_loss:.4f} | "
          f"acc={acc:.4f} | bacc={bacc:.4f} | auc={np.mean(list(auc_per_class.values())):.4f}", end="")

    scheduler.step()

    if bacc > best_bacc:
        best_bacc, best_epoch = bacc, epoch
        no_improve = 0
        torch.save(model.state_dict(), ckpt_path)
        print("  ✓ saved")
    else:
        no_improve += 1
        print(f"  (no improve {no_improve}/{Config['patience']})")
        if no_improve >= Config["patience"]:
            print(f"\nEarly stopping at epoch {epoch}.")
            break

print(f"\nDone in {(time.time()-t0)/60:.1f} min")

print(f"Epoch {epoch:3d}/{Config['epochs']} | loss={train_loss:.4f} | "
      f"acc={acc:.4f} | bacc={bacc:.4f} | "
      f"sens={np.mean(sensitivity):.4f} | spec={np.mean(specificity):.4f} | "
      f"auc={np.mean(list(auc_per_class.values())):.4f}", end="")

history_path = os.path.join(CKPT_DIR, "history.json")
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)
print(f"History saved to {history_path}")

Training | pretrained=True | loss=weighted_ce

Epoch   1/30 | loss=0.9355 | acc=0.7462 | bacc=0.4962 | auc=0.9403  ✓ saved


***!!IGNORE THE LINES BELOW FOR NOW***

In [ ]:
#wandb.login(key="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")
#run the below for W&B login
#!/home/umw7eg/.local/bin/wandb login
#wandb.login(key="API_key")

wandb.init(
    entity="umw7eg_uva",
    project="ISIC2019",
    name=f"{ARCHITECTURE}_{LOSS_FN}_{AUGMENTATION}_pretrained{PRETRAINED}",
    config={
        "architecture":  ARCHITECTURE,
        "loss_fn":       LOSS_FN,
        "augmentation":  AUGMENTATION,
        "pretrained":    PRETRAINED,
        "freeze_bb":     FREEZE_BB,
        "img_size":      IMG_SIZE,
        "batch_size":    BATCH_SIZE,
        "epochs":        EPOCHS,
        "lr":            LR,
        "val_split":     VAL_SPLIT,
        "patience":      PATIENCE,
        "seed":          SEED,
    }
)

## Patient-Grouped Train / Val Split

In [8]:
def make_split(csv_path, meta_path):
    """
    GroupShuffleSplit guarantees all images with the same lesion_id
    land in the same fold. This prevents patient data leakage.
    """
    df   = pd.read_csv(csv_path)
    meta = pd.read_csv(meta_path)
    df   = df.merge(meta[["image", "lesion_id"]], on="image", how="left")
    df["lesion_id"] = df["lesion_id"].fillna(df["image"])
    print("Patient-grouped split using lesion_id")
    
    df["label"] = df[CLASSES].values.argmax(axis=1)
    
    gss   = GroupShuffleSplit(1, test_size=VAL_SPLIT, random_state=SEED)
    ti, vi = next(gss.split(df, df["label"], groups=df["lesion_id"]))
    return df.iloc[ti].copy(), df.iloc[vi].copy()

train_df, val_df = make_split(TRAIN_CSV, TRAIN_META)
test_df = pd.read_csv(TEST_CSV)
if "UNK" in test_df.columns:
    test_df = test_df[test_df["UNK"] != 1].copy()

# 10% subset ── TO BE REMOVED
train_df = train_df.sample(frac=0.1, random_state=SEED).reset_index(drop=True)
val_df   = val_df.sample(frac=0.1,   random_state=SEED).reset_index(drop=True)
test_df  = test_df.sample(frac=0.1,  random_state=SEED).reset_index(drop=True)
print(f"Subset — train: {len(train_df)}, val: {len(val_df)}, test: {len(test_df)}")

# Build datasets & dataloaders
train_ds = ISICDataset(train_df, TRAIN_DIR, get_transforms("train"))
val_ds   = ISICDataset(val_df,   TRAIN_DIR, get_transforms("val"))
test_ds  = ISICDataset(test_df,  TEST_DIR,  get_transforms("test"))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# Print class distribution
for name, df_ in [("TRAIN", train_df), ("VAL", val_df)]:
    labels = df_[CLASSES].values.argmax(axis=1)
    cnts   = np.bincount(labels, minlength=NUM_CLASSES)
    
    print(f"\n{name} ({len(df_)} images):")
    for i, c in enumerate(CLASSES):
        print(f"  {c:6s}: {cnts[i]:5d}  ({100*cnts[i]/len(df_):4.1f}%)")

Patient-grouped split using lesion_id
Subset — train: 2015, val: 518, test: 619

TRAIN (2015 images):
  MEL   :   347  (17.2%)
  NV    :  1005  (49.9%)
  BCC   :   302  (15.0%)
  AK    :    64  ( 3.2%)
  BKL   :   197  ( 9.8%)
  DF    :    23  ( 1.1%)
  VASC  :    20  ( 1.0%)
  SCC   :    57  ( 2.8%)

VAL (518 images):
  MEL   :    97  (18.7%)
  NV    :   268  (51.7%)
  BCC   :    60  (11.6%)
  AK    :    26  ( 5.0%)
  BKL   :    54  (10.4%)
  DF    :     1  ( 0.2%)
  VASC  :     3  ( 0.6%)
  SCC   :     9  ( 1.7%)


## Loss, Optimizer, Scheduler

In [9]:

def get_loss():
    freqs   = np.array([CLASS_FREQ[c] for c in CLASSES])
    weights = torch.tensor(1.0 / (NUM_CLASSES * freqs), dtype=torch.float32)
    weights = weights / weights.sum() * NUM_CLASSES
    
    if LOSS_FN == "ce":
        return nn.CrossEntropyLoss().to(device)
    elif LOSS_FN == "weighted_ce":
        return nn.CrossEntropyLoss(weight=weights.to(device))
    elif LOSS_FN == "focal":
        class FocalLoss(nn.Module):
            def __init__(self, gamma=2.0, alpha=None):
                super().__init__()
                self.gamma = gamma
                self.alpha = alpha
            def forward(self, logits, targets):
                ce = F.cross_entropy(logits, targets, reduction="none")
                pt = F.softmax(logits, 1).gather(1, targets.unsqueeze(1)).squeeze(1)
                at = self.alpha.to(logits.device)[targets]
                return (at * (1 - pt) ** self.gamma * ce).mean()
        return FocalLoss(gamma=2.0, alpha=weights).to(device)

loss_fn   = get_loss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=LR, weight_decay=1e-2)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

total = sum(p.numel() for p in model.parameters())
print(f"Model: EfficientNet-B0 | Params: {total:,} | Device: {device}")
print(f"Loss: {LOSS_FN} | Pretrained: {PRETRAINED}")

Model: EfficientNet-B0 | Params: 4,017,796 | Device: cuda
Loss: weighted_ce | Pretrained: False


## Benchmarking (Params, Size, FLOPs, Inference Time)

In [11]:
def count_parameters(m):
    total     = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    return total, trainable

def get_model_size_mb(m):
    param_size  = sum(p.nelement() * p.element_size() for p in m.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in m.buffers())
    return (param_size + buffer_size) / (1024 ** 2)

def measure_inference_time(m, input_shape=(1, 3, 224, 224), num_runs=100):
    """Average inference time in ms over num_runs forward passes."""
    m.eval()
    dev   = next(m.parameters()).device
    dummy = torch.randn(input_shape).to(dev)
    for _ in range(10):          # warm-up
        with torch.no_grad():
            _ = m(dummy)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    with torch.no_grad():
        for _ in range(num_runs):
            _ = m(dummy)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return (time.time() - t0) / num_runs * 1000.0

def estimate_flops(m, input_shape=(1, 3, 224, 224)):
    """Estimate FLOPs via forward hooks on Conv2d and Linear layers."""
    m.eval()
    flops_total = 0
    handles     = []

    def conv_hook(mod, inp, out):
        nonlocal flops_total
        N = inp[0].shape[0]
        Hout, Wout = out.shape[-2], out.shape[-1]
        Kh, Kw = (mod.kernel_size if isinstance(mod.kernel_size, tuple)
                  else (mod.kernel_size, mod.kernel_size))
        flops = 2 * N * (mod.in_channels // mod.groups) * Kh * Kw * mod.out_channels * Hout * Wout
        if mod.bias is not None:
            flops += N * mod.out_channels * Hout * Wout
        flops_total += int(flops)

    def linear_hook(mod, inp, out):
        nonlocal flops_total
        batch = int(inp[0].numel() // mod.in_features)
        flops = 2 * batch * mod.in_features * mod.out_features
        if mod.bias is not None:
            flops += batch * mod.out_features
        flops_total += int(flops)

    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            handles.append(mod.register_forward_hook(conv_hook))
        elif isinstance(mod, nn.Linear):
            handles.append(mod.register_forward_hook(linear_hook))

    dummy = torch.randn(input_shape).to(next(m.parameters()).device)
    with torch.no_grad():
        m(dummy)
    for h in handles:
        h.remove()
    return flops_total

def print_benchmark(m, label="EfficientNet-B0"):
    total_p, trainable_p = count_parameters(m)
    size_mb = get_model_size_mb(m)
    inf_ms  = measure_inference_time(m, input_shape=(1, 3, IMG_SIZE, IMG_SIZE))
    flops   = estimate_flops(m, input_shape=(1, 3, IMG_SIZE, IMG_SIZE))
    print(f"\n── BENCHMARK: {label} ──")
    print(f"  Total params      : {total_p:,}")
    print(f"  Trainable params  : {trainable_p:,}")
    print(f"  Model size        : {size_mb:.2f} MB")
    print(f"  Inference time    : {inf_ms:.2f} ms  (single image, avg over 100 runs)")
    print(f"  FLOPs (forward)   : {flops/1e9:.3f} GFLOPs")
    return {"label": label, "params_total": total_p, "params_trainable": trainable_p,
            "size_mb": size_mb, "inference_ms": inf_ms, "gflops": flops / 1e9}

print("Benchmark functions defined")

Benchmark functions defined


## Training Loop (30 epochs, early stopping)

In [12]:
best_bacc, best_epoch, no_improve = -1, 0, 0
history   = []
ckpt_path = os.path.join(CKPT_DIR, f"efficientnet_b0_{LOSS_FN}_{AUGMENTATION}.pt")

print(f"Training EfficientNet-B0 | loss={LOSS_FN} | aug={AUGMENTATION} | pretrained={PRETRAINED}\n")
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    run_loss, correct, total = 0.0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        run_loss += loss.item() * imgs.size(0)
        correct  += (logits.argmax(1) == labels).sum().item()
        total    += imgs.size(0)

    scheduler.step()
    train_loss = run_loss / total
    train_acc  = correct / total

    # ── Validate ──
    m = evaluate(model, val_loader)
    if epoch % 5 == 0:
        print_metrics(m, split="val")

    history.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc,
                    "val_bacc": m["bacc"], "val_auc": m["auc_macro"], "val_f1": m["f1_macro"]})

    # ── W&B per-epoch logging ──
    log_dict = {
        "epoch":      epoch,
        "train/loss": train_loss,
        "train/acc":  train_acc,
        "val/bacc":   m["bacc"],
        "val/auc":    m["auc_macro"],
        "val/f1":     m["f1_macro"],
    }
    for c in CLASSES:
        log_dict[f"val/auc_{c}"] = m["per_auc"][c]
    #wandb.log(log_dict)

    print(f"Epoch {epoch:3d}/{EPOCHS} | loss={train_loss:.4f} | acc={train_acc:.4f} | "
          f"val BACC={m['bacc']:.4f} | AUC={m['auc_macro']:.4f} | F1={m['f1_macro']:.4f}", end="")

    if m["bacc"] > best_bacc:
        best_bacc, best_epoch = m["bacc"], epoch
        no_improve = 0
        torch.save(model.state_dict(), ckpt_path)
        print("  saved")
    else:
        no_improve += 1
        print(f"  (no improve {no_improve}/{PATIENCE})")
        if no_improve >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}.")
            break

print(f"\nDone in {(time.time()-t0)/60:.1f} min")
print(f"Best val BACC: {best_bacc:.4f} @ epoch {best_epoch}")

# Save history locally as backup
history_path = os.path.join(CKPT_DIR, f"history_{LOSS_FN}_{AUGMENTATION}.json")
with open(history_path, "w") as f:
    json.dump(history, f, indent=2)
print(f"History saved to {history_path}")

Training EfficientNet-B0 | loss=weighted_ce | aug=strong | pretrained=False



Error: You must call wandb.init() before wandb.log()

## Post-Training Benchmark & W&B Summary

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(ckpt_path, map_location=device))
bench = print_benchmark(model, label=f"EfficientNet-B0 ({LOSS_FN}, aug={AUGMENTATION})")

# Push to W&B summary (visible in runs table)
wandb.summary["benchmark/params_total"]     = bench["params_total"]
wandb.summary["benchmark/params_trainable"] = bench["params_trainable"]
wandb.summary["benchmark/size_mb"]          = bench["size_mb"]
wandb.summary["benchmark/inference_ms"]     = bench["inference_ms"]
wandb.summary["benchmark/gflops"]           = bench["gflops"]
wandb.summary["best_val_bacc"]              = best_bacc
wandb.summary["best_epoch"]                 = best_epoch

# Upload checkpoint as W&B artifact
artifact = wandb.Artifact(
    name=f"efficientnet_b0_{LOSS_FN}_{AUGMENTATION}",
    type="model",
    description=f"Best checkpoint — val BACC {best_bacc:.4f} @ epoch {best_epoch}"
)
artifact.add_file(ckpt_path)
wandb.log_artifact(artifact)
print("Checkpoint artifact logged to W&B")

wandb.finish()
print("W&B run finished.")